# Practical RAG Hyperparameter Tuning Workflow for Elrond Investigation

In this notebook, we'll test different RAG settings and compare the results to find the best combination. This workflow is based on my Elrond_Investigation_RAG_Pipeline.ipynb notebook, please refer to it if you need more infromation:
https://github.com/MariyaSha/rag_ollama/blob/main/Elrond_Investigation_RAG_Pipeline.ipynb

## Import Modules

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

## Select Hyperparameters to Tune
Define the values we want to test, finsing their best combination.

In [3]:
chunk_sizes = [500, 600, 700]
chunk_overlaps = [50, 150, 250]
ks = [3, 5, 7]

## Load Documents and Models
Load anything that is not affected by the hyperparameters - the documents, embeddings, language model, and question. Ignore every other component in the workflow for now, as it directly depends on `chunk_size`, `chunk_overlap` and `k` values.

In [10]:
file_names = sorted(os.listdir("data"))

all_pages = []

for file in file_names:
    loader = PyPDFLoader("data/" + file)
    pages = loader.load()
    all_pages.extend(pages)

llm = ChatOllama(model="qwen2.5:1.5b")
embeddings = OllamaEmbeddings(model="bge-m3")

question = "Why is Elrond under investigation?"

## Rebuild the Ask Function
We must adjust the `ask()` function to receive retrievers with changing `k` values.

In [6]:
def ask_with_retriever(question, retriever):
    retrieved_chunks = retriever.invoke(question)

    context = ""

    for chunk in retrieved_chunks:
        context += chunk.page_content + "\n\n"

    response = llm.invoke(
        f"""
        You are an AI detective investigating whether Elrond is Agent Smith.

        Answer ONLY using the provided context.

        If the answer cannot be found in the context, say:
        "I don't know based on the case files."

        Context:
        {context}

        Question:
        {question}
        """
    )

    return response.content, retrieved_chunks

## Hyperparameter Tuning Loop
Build a new vector database for each `chunk_size` and `chunk_overlap` setting, test them on different retrievers with changing `k` values.
In the end of the process, we will have `3 * 3 * 3` or `27` different responses from the model, generated with different combinations of parameters.

In [11]:
results = []

for chunk_size in chunk_sizes:
    for chunk_overlap in chunk_overlaps:

        print("=" * 50)
        print(f"""
            Building vector DB...
            chunk_size={chunk_size}
            chunk_overlap={chunk_overlap}
        """)
        print("=" * 50)

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        chunks = splitter.split_documents(all_pages)
        vector_db = FAISS.from_documents(chunks, embeddings)

        for k in ks:
            retriever = vector_db.as_retriever(
                search_kwargs={"k": k}
            )

            answer, retrieved_chunks = ask_with_retriever(question, retriever)

            result = {
                "chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap,
                "k": k,
                "num_chunks": len(chunks),
                "answer": answer
            }

            results.append(result)

            print("-" * 50)
            print(f"""
                chunk_size={chunk_size}
                chunk_overlap={chunk_overlap}
                k={k}
                """)
            print("-" * 50)
            print(answer)
            print()


            Building vector DB...
            chunk_size=500
            chunk_overlap=50
        
--------------------------------------------------

                chunk_size=500
                chunk_overlap=50
                k=3
                
--------------------------------------------------
Elrond is under investigation because he is suspected of being associated with an extradimensional entity known as "Agent Smith," which is the primary allegation in the case summary provided. This suspicion stems from the fact that Galadriel believes it would be beneficial to have more discussion about art and less on cutlery in Lord Elrond's report, indicating a focus on his artistic talents rather than other factors typically considered in such investigations.

--------------------------------------------------

                chunk_size=500
                chunk_overlap=50
                k=5
                
--------------------------------------------------
Elrond is under investig

## Evaluate Results

### Reference answer
Before looking at any of the generated answers, write the answer you would expect to receive. Keep it as short and direct as you'd like it to be. We'll use this as the reference when comparing all 27 responses.

In [13]:
reference_answer = """
Elrond is under investigation because witness statements reported irregular behavior,
leading to allegations that he may have possible affiliation with, impersonation by,
or identity overlap with an extradimensional entity known as Agent Smith.
"""

### Score Answers From Results
Give scores to each answer from the results using similarity metrics from Sklearn.

#### Install Additional Modules

In [19]:
!pip install scikit-learn
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 59.2 MB/s  0:00:00


#### Define Evaluation Function

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def evaluate_answer(reference, answer):
    texts = [reference, answer]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )

    tfidf = vectorizer.fit_transform(texts)

    similarity = cosine_similarity(tfidf[0], tfidf[1])[0][0]

    ref_words = reference.split()
    answer_words = answer.split()

    length_ratio = min(len(answer_words), len(ref_words)) / max(len(answer_words), len(ref_words))

    return {
        "similarity": similarity,
        "length_ratio": length_ratio,
        "final_score": (similarity * 0.8) + (length_ratio * 0.2)
    }

#### Evaluate
Compare every answer to the reference answer and sort the results from best to worst.

In [22]:
scored_results = []

for result in results:
    scores = evaluate_answer(reference_answer, result["answer"])

    scored_results.append({
        **result,
        **scores
    })

df = pd.DataFrame(scored_results)

df = df.sort_values("final_score", ascending=False)

df[[
    "chunk_size",
    "chunk_overlap",
    "k",
    "similarity",
    "length_ratio",
    "final_score",
    "answer"
]]

,chunk_size,chunk_overlap,k,similarity,length_ratio,final_score,answer
21,700,150,3,0.601253,0.825000,0.646002,Elrond is under investigation due to the alleg...
13,600,150,5,0.549133,0.733333,0.585973,Elrond is under investigation due to allegatio...
12,600,150,3,0.423066,0.942857,0.527024,Elrond is under investigation because of possi...
16,600,250,5,0.411338,0.846154,0.498301,Elrond is under investigation due to the asser...
9,600,50,3,0.411338,0.804878,0.490046,Elrond is under investigation due to the prima...
23,700,150,7,0.432960,0.717391,0.489846,Elrond is under investigation because he has b...
20,700,50,7,0.378592,0.804878,0.463850,"Based on the provided context, Elrond is under..."
1,500,50,5,0.376170,0.767442,0.454424,Elrond is under investigation because someone ...
18,700,50,3,0.335518,0.666667,0.401748,Elrond is under investigation due to the alleg...
8,500,250,7,0.328019,0.673469,0.397109,Elrond is under investigation because of a pri...


## Set Winning Hyperparameters
## Set the Winning Hyperparameters

Use the best-performing combination as the default settings for the remainder of the notebook.
Perfect! now we know for sure that the 21st combination we tried is the winning one. Now if we go back to our previous notebook:
https://github.com/MariyaSha/rag_ollama/blob/main/Elrond_Investigation_RAG_Pipeline.ipynb

Ans we set it's chunk_size, chunk_overlap and k values to the numbers below - we will get the best results.

In [23]:
best = df.iloc[0]

print("BEST CONFIG")
print("=" * 50)
print(f"chunk_size: {best['chunk_size']}")
print(f"chunk_overlap: {best['chunk_overlap']}")
print(f"k: {best['k']}")
print(f"similarity: {best['similarity']:.3f}")
print(f"length_ratio: {best['length_ratio']:.3f}")
print(f"final_score: {best['final_score']:.3f}")
print()
print(best["answer"])

BEST CONFIG
chunk_size: 700
chunk_overlap: 150
k: 3
similarity: 0.601
length_ratio: 0.825
final_score: 0.646

Elrond is under investigation due to the allegations of possible affiliation, impersonation by, or identity overlap with an extradimensional entity known as "Agent Smith." The context indicates that this was initiated following witness statements about irregular behavior by Lord Elrond.


## Congradulations!
### Now you know how to tune parameters as well! :)